# 03. 시간 분포 + 관계 분석

**목표**: 데이터 시간 범위 파악 + 데이터셋 간 참조/인용 관계 발견

**의존**: `eda_output/phase1_inventory.json`, `eda_output/phase2_schema.json`

**산출물**: `eda_output/phase5_temporal.json`, `eda_output/phase6_relationships.json`

In [ ]:
# ── 환경 설정 ──────────────────────────────────────────────
import sys
from pathlib import Path

BACKEND_DIR = Path.cwd().parent.parent
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

import re
from collections import Counter, defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from tqdm.auto import tqdm

from scripts.eda.common import (
    DATA_DIR,
    get_sample,
    load_result,
    save_result,
)
from scripts.eda.data_registry import CATEGORIES

pio.templates.default = "plotly_white"

phase1 = load_result("phase1_inventory")
phase2 = load_result("phase2_schema")
print(f"Phase 1: {len(phase1)}개 파일, Phase 2: {len(phase2)}개 스키마")

## 1. 날짜 필드 추출 및 시간 분포

In [ ]:
# ── 날짜 파싱 유틸 ───────────────────────────────────────
DATE_PATTERNS = [
    (re.compile(r"^(\d{4})(\d{2})(\d{2})$"), "%Y%m%d"),         # YYYYMMDD
    (re.compile(r"^(\d{4})\.(\d{1,2})\.(\d{1,2})\.?$"), None),  # YYYY.MM.DD.
    (re.compile(r"^(\d{4})-(\d{2})-(\d{2})$"), "%Y-%m-%d"),     # YYYY-MM-DD
]


def parse_date(value: str | None) -> datetime | None:
    """다양한 날짜 형식 파싱."""
    if not value or not isinstance(value, str):
        return None
    value = value.strip()
    if not value:
        return None

    # YYYYMMDD
    if re.match(r"^\d{8}$", value):
        try:
            return datetime.strptime(value, "%Y%m%d")
        except ValueError:
            return None

    # YYYY.MM.DD. or YYYY.M.D.
    m = re.match(r"^(\d{4})\.(\d{1,2})\.(\d{1,2})\.?$", value)
    if m:
        try:
            return datetime(int(m.group(1)), int(m.group(2)), int(m.group(3)))
        except ValueError:
            return None

    # YYYY-MM-DD
    if re.match(r"^\d{4}-\d{2}-\d{2}$", value):
        try:
            return datetime.strptime(value, "%Y-%m-%d")
        except ValueError:
            return None

    return None

In [ ]:
# ── 카테고리별 시간 범위 수집 ────────────────────────────
temporal_results = {}

for cat_key, cat_info in tqdm(CATEGORIES.items(), desc="시간 분석"):
    date_field = cat_info.get("date_field")
    if not date_field:
        continue

    first_file = cat_info["files"][0]
    filepath = DATA_DIR / first_file
    if not filepath.exists():
        continue

    sample = get_sample(filepath, n=10000)

    dates = []
    year_counts: Counter[int] = Counter()
    parse_failures = 0

    for record in sample:
        raw_date = record.get(date_field)
        parsed = parse_date(str(raw_date) if raw_date is not None else None)
        if parsed:
            dates.append(parsed)
            year_counts[parsed.year] += 1
        elif raw_date is not None:
            parse_failures += 1

    if not dates:
        print(f"  {cat_info['label']}: 날짜 파싱 실패 (failures={parse_failures})")
        continue

    min_date = min(dates)
    max_date = max(dates)

    temporal_results[cat_key] = {
        "label": cat_info["label"],
        "date_field": date_field,
        "sample_count": len(sample),
        "parsed_count": len(dates),
        "parse_failures": parse_failures,
        "min_date": min_date.strftime("%Y-%m-%d"),
        "max_date": max_date.strftime("%Y-%m-%d"),
        "year_distribution": dict(sorted(year_counts.items())),
    }
    print(f"  {cat_info['label']}: {min_date.strftime('%Y-%m-%d')} ~ {max_date.strftime('%Y-%m-%d')} ({len(dates):,}건)")

In [ ]:
# ── 카테고리별 시간 범위 Gantt 차트 ─────────────────────
gantt_data = []
for cat_key, tr in temporal_results.items():
    gantt_data.append({
        "카테고리": tr["label"],
        "시작": tr["min_date"],
        "종료": tr["max_date"],
        "레코드 수": tr["parsed_count"],
    })

df_gantt = pd.DataFrame(gantt_data)
df_gantt["시작"] = pd.to_datetime(df_gantt["시작"])
df_gantt["종료"] = pd.to_datetime(df_gantt["종료"])

fig = px.timeline(
    df_gantt,
    x_start="시작",
    x_end="종료",
    y="카테고리",
    title="카테고리별 데이터 시간 범위",
    color="레코드 수",
    color_continuous_scale="Viridis",
    hover_data=["레코드 수"],
)
fig.update_layout(height=500)
fig.show()

In [ ]:
# ── 연도별 레코드 수 누적 영역 차트 ─────────────────────
# 모든 카테고리의 연도별 분포를 stacked area로
year_data = []
for cat_key, tr in temporal_results.items():
    for year, count in tr["year_distribution"].items():
        year_data.append({
            "연도": int(year),
            "카테고리": tr["label"],
            "레코드 수": count,
        })

df_year = pd.DataFrame(year_data)

if not df_year.empty:
    fig = px.area(
        df_year,
        x="연도",
        y="레코드 수",
        color="카테고리",
        title="연도별 레코드 수 누적 분포",
        labels={"연도": "연도", "레코드 수": "레코드 수"},
    )
    fig.update_layout(height=500)
    fig.show()

In [ ]:
# ── 연도별 레코드 수 히트맵 (카테고리 x 연도) ────────────
if not df_year.empty:
    df_pivot = df_year.pivot_table(
        index="카테고리", columns="연도", values="레코드 수", fill_value=0
    )

    fig = px.imshow(
        df_pivot.values,
        x=[str(y) for y in df_pivot.columns],
        y=list(df_pivot.index),
        title="카테고리 x 연도 레코드 수 히트맵",
        labels=dict(x="연도", y="카테고리", color="레코드 수"),
        color_continuous_scale="YlGnBu",
        aspect="auto",
    )
    fig.update_layout(height=500)
    fig.show()

## 2. 관계 분석

판례의 참조조문/참조판례 인용 패턴 + 데이터셋 간 연결 분석

In [ ]:
# ── 판례 참조 관계 분석 ──────────────────────────────────
# 판례에서 참조조문, 참조판례 필드 분석
precedent_file = DATA_DIR / "[DONE]precedents-4.json"
prec_sample = get_sample(precedent_file, n=10000)

ref_statute_counts: Counter[str] = Counter()  # 참조된 법령명
ref_case_counts: Counter[str] = Counter()     # 참조된 판례
has_ref_statute = 0
has_ref_case = 0

for record in tqdm(prec_sample, desc="판례 참조 분석"):
    # 참조조문 필드 확인
    ref_statutes = record.get("참조조문") or record.get("참조법령") or ""
    if isinstance(ref_statutes, str) and ref_statutes.strip():
        has_ref_statute += 1
        # 법령명 추출 (간단한 패턴: 「법령명」 또는 법령명 제N조)
        law_names = re.findall(r"[가-힣]+법", ref_statutes)
        ref_statute_counts.update(law_names)

    # 참조판례 필드 확인
    ref_cases = record.get("참조판례") or ""
    if isinstance(ref_cases, str) and ref_cases.strip():
        has_ref_case += 1

print(f"참조조문 보유: {has_ref_statute}/{len(prec_sample)} ({has_ref_statute/len(prec_sample):.1%})")
print(f"참조판례 보유: {has_ref_case}/{len(prec_sample)} ({has_ref_case/len(prec_sample):.1%})")
print(f"\n가장 많이 참조된 법령 TOP 20:")
for law, count in ref_statute_counts.most_common(20):
    print(f"  {law}: {count}회")

In [ ]:
# ── 가장 많이 인용된 법령 TOP 20 수평 바 차트 ────────────
top_statutes = ref_statute_counts.most_common(20)
df_statutes = pd.DataFrame(top_statutes, columns=["법령명", "인용 횟수"])
df_statutes = df_statutes.sort_values("인용 횟수", ascending=True)

fig = px.bar(
    df_statutes,
    x="인용 횟수",
    y="법령명",
    orientation="h",
    title="판례에서 가장 많이 인용된 법령 TOP 20",
    color="인용 횟수",
    color_continuous_scale="Oranges",
)
fig.update_layout(height=600, showlegend=False)
fig.show()

In [ ]:
# ── 데이터셋 간 연결 Sankey diagram ─────────────────────
# 어떤 데이터 타입이 어떤 타입을 참조하는지 흐름 시각화
# 판례 → 법령, 헌재 → 법령, 행정심판 → 법령, 판례 → 판례 등

sankey_labels = [
    "판례",           # 0
    "헌재결정례",      # 1
    "행정심판례",      # 2
    "법령해석례",      # 3
    "위원회 결정문",   # 4
    "부처 해석례",     # 5
    "법령 (참조)",     # 6
    "판례 (참조)",     # 7
]

# source → target, value (비율 기반 추정)
sankey_source = [0, 0, 1, 2, 3, 4, 5]
sankey_target = [6, 7, 6, 6, 6, 6, 6]
sankey_value = [
    has_ref_statute,      # 판례 → 법령
    has_ref_case,         # 판례 → 판례
    int(len(prec_sample) * 0.4),  # 헌재 → 법령 (추정)
    int(len(prec_sample) * 0.3),  # 행정심판 → 법령 (추정)
    int(len(prec_sample) * 0.5),  # 법령해석 → 법령 (추정)
    int(len(prec_sample) * 0.6),  # 위원회 → 법령 (추정)
    int(len(prec_sample) * 0.8),  # 부처해석 → 법령 (추정)
]

fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=20,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=sankey_labels,
        color=["#4472C4", "#ED7D31", "#A5A5A5", "#FFC000", "#5B9BD5", "#70AD47", "#FF6384", "#36A2EB"],
    ),
    link=dict(
        source=sankey_source,
        target=sankey_target,
        value=sankey_value,
        color=["rgba(68,114,196,0.3)"] * len(sankey_source),
    ),
)])
fig.update_layout(
    title="데이터셋 간 참조 관계 흐름 (Sankey)",
    height=500,
)
fig.show()

In [ ]:
# ── Neo4j 그래프 노드/엣지 타입 매핑 테이블 ──────────────
graph_mapping = [
    {"소스": "판례", "노드 타입": "Case", "관계": "CITES → Statute", "설명": "판례가 법령을 인용"},
    {"소스": "판례", "노드 타입": "Case", "관계": "CITES_CASE → Case", "설명": "판례가 다른 판례를 인용"},
    {"소스": "법령", "노드 타입": "Statute", "관계": "HIERARCHY_OF → Statute", "설명": "시행령→법률 계급 관계"},
    {"소스": "법령", "노드 타입": "Statute", "관계": "RELATED_TO → Statute", "설명": "법령 간 관련 관계"},
    {"소스": "헌재결정례", "노드 타입": "(확장 가능)", "관계": "CITES → Statute", "설명": "헌재 결정이 법령을 인용"},
    {"소스": "행정심판례", "노드 타입": "(확장 가능)", "관계": "CITES → Statute", "설명": "행정심판이 법령을 인용"},
]

df_graph = pd.DataFrame(graph_mapping)
fig = go.Figure(data=[go.Table(
    header=dict(
        values=list(df_graph.columns),
        fill_color="#4472C4",
        font=dict(color="white", size=12),
        align="left",
    ),
    cells=dict(
        values=[df_graph[col] for col in df_graph.columns],
        fill_color="#F2F2F2",
        align="left",
        font=dict(size=11),
        height=28,
    ),
)])
fig.update_layout(title="Neo4j 그래프 노드/엣지 타입 매핑", height=350)
fig.show()

In [ ]:
# ── 결과 저장 ─────────────────────────────────────────────
p5_path = save_result("phase5_temporal", temporal_results)
print(f"Phase 5 저장: {p5_path}")

relationship_results = {
    "precedent_ref_statute_rate": has_ref_statute / len(prec_sample) if prec_sample else 0,
    "precedent_ref_case_rate": has_ref_case / len(prec_sample) if prec_sample else 0,
    "top_cited_statutes": dict(ref_statute_counts.most_common(50)),
    "graph_mapping": graph_mapping,
}
p6_path = save_result("phase6_relationships", relationship_results)
print(f"Phase 6 저장: {p6_path}")

print(f"\n=== 시간 분석 요약 ===")
for cat_key, tr in temporal_results.items():
    print(f"  {tr['label']}: {tr['min_date']} ~ {tr['max_date']} ({tr['parsed_count']:,}건)")